In [6]:
import pandas as pd 
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np 

from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D,Dropout,Flatten,Dense, Rescaling
from tensorflow.keras.utils import image_dataset_from_directory


In [3]:
batch_size = 32
height = 150
width = 150

In [2]:
trainPath = '/Users/bhavesh/Documents/GitHub/cognition-3.0/Final-Project/archive/seg_train'
testPath = '/Users/bhavesh/Documents/GitHub/cognition-3.0/Final-Project/archive/seg_test'
predPath = '/Users/bhavesh/Documents/GitHub/cognition-3.0/Final-Project/archive/seg_pred'

In [10]:
def data(path,labels):
    dataset = image_dataset_from_directory(
        directory = path,
        labels = labels,
        seed = 123,
        image_size=(height,width),
        batch_size = batch_size
    )
    return dataset

In [11]:
trainDS = data(trainPath, labels='inferred')
testDS = data(testPath, labels='inferred')
predDs = data(predPath, labels=None)

Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
Found 7301 files.


In [12]:
className = trainDS.class_names
print(className)

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [14]:
standard = Rescaling(1./255)  #normalizing 

trainDS = trainDS.map(lambda x,y: (standard(x), y))
testDS = testDS.map(lambda x,y: (standard(x), y))
predDs = predDs.map(lambda x: (standard(x)))

In [16]:
model = Sequential([
     tf.keras.Input(shape=(150,150,3)),
     Conv2D(16,(3,3), activation='relu'),
     MaxPooling2D(2,2),
     Conv2D(32,(3,3), activation='relu'),
     MaxPooling2D(2,2),
     Conv2D(64,(3,3), activation='relu'),
     MaxPooling2D(2,2),
     Flatten(),
     Dense(128, activation='relu'),
     Dense(len(className),activation='softmax')

])

In [17]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,367,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,391,974 (9.12 MB)

 Trainable params: 2,391,974 (9.12 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
epochs = 10
history = model.fit(trainDS, validation_data=testDS, epochs=epochs)

Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 118s 266ms/step - accuracy: 0.5351 - loss: 1.1783 - val_accuracy: 0.7287 - val_loss: 0.7288
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 117s 266ms/step - accuracy: 0.7508 - loss: 0.6923 - val_accuracy: 0.7500 - val_loss: 0.7085
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 115s 261ms/step - accuracy: 0.8091 - loss: 0.5327 - val_accuracy: 0.7893 - val_loss: 0.6121
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 122s 277ms/step - accuracy: 0.8583 - loss: 0.4015 - val_accuracy: 0.7773 - val_loss: 0.6771
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 115s 263ms/step - accuracy: 0.8961 - loss: 0.3035 - val_accuracy: 0.7857 - val_loss: 0.7112
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 122s 278ms/step - accuracy: 0.9307 - loss: 0.2076 - val_accuracy: 0.7930 - val_loss: 0.6699
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 115s 263ms/step - accuracy: 0.9498 - loss: 0.1526 - val_accuracy: 0.8067 - val_loss: 0.7975
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 116s 264ms/step - accuracy: 0.9667 -

In [21]:
print('hi')

hi


In [22]:
model.save('final.keras') # type: ignore
print("Model saved to final.keras")

Model saved to final.keras


In [23]:
model_loss, model_accuracy = model.evaluate(testDS, verbose=2)

94/94 - 8s - 81ms/step - accuracy: 0.7880 - loss: 1.1221


In [29]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the saved model
model = load_model('final.keras')

# Define image size
height = 150
width = 150

# Function to preprocess the image
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(height, width))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Create a batch
    img_array = img_array / 255.0  # Normalize the image
    return img_array

# Function to predict the class of the image
def predict_image_class(img_path):
    img_array = preprocess_image(img_path)
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions, axis=1)
    return predicted_class[0]

# Provide the path to your image
image_path = '/Users/bhavesh/Documents/GitHub/cognition-3.0/Final-Project/archive/seg_pred/23099.jpg'

# Predict the class
predicted_class = predict_image_class(image_path)

# Print the predicted class name
class_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
print(f"The predicted class is: {class_names[predicted_class]}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
The predicted class is: sea
